# LLM prompts used
-how to Train a pretrained model on the rag-mini-bioasq dataset

-how to Calculate the ROUGE-L score and BERT-F1 score

# Preprocessing

In [1]:
import pandas as pd

passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_data = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

print(passages.head())
print(test_data.head())


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
23188  Kinetic and electrophoretic properties of 230-...
23469  Male Wistar specific-pathogen-free rats aged 2...
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   
3                   Are long non coding RNAs spliced?   
4                   Is RANKL secreted from the cells?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  

In [2]:
import pandas as pd

# Load passages
passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

# Show basic info
print(passages.info())
print("\nSample rows:\n", passages.head(5))


<class 'pandas.core.frame.DataFrame'>
Index: 40221 entries, 9797 to 34894461
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   passage  40221 non-null  string
dtypes: string(1)
memory usage: 628.5 KB
None

Sample rows:
                                                  passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
23188  Kinetic and electrophoretic properties of 230-...
23469  Male Wistar specific-pathogen-free rats aged 2...


In [3]:
import re
import pandas as pd

# Load the passages dataset
passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

print("Before Preprocessing Sample:\n", passages.head(3))

# Step 1: Remove duplicates
passages = passages.drop_duplicates(subset=['passage'])

# Step 2: Define cleaning function
def clean_passage(text):
    text = text.strip().lower()  # Convert to lowercase
    text = re.sub(r"[^a-z0-9.,;:()\-/% ]", " ", text)  # Remove unwanted characters
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    return text

# Step 3: Apply cleaning
passages['passage'] = passages['passage'].astype(str).apply(clean_passage)

# Step 4: Drop empty passages
passages = passages[passages['passage'].str.strip() != ""]

# Step 5: Show final result
print("\nAfter Preprocessing Sample:\n", passages.head(3))
print("\nTotal passages after cleaning:", passages.shape)


Before Preprocessing Sample:
                                                  passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...

After Preprocessing Sample:
                                                  passage
id                                                      
9797   new data on viruses isolated from patients wit...
11906  we describe an improved method for detecting d...
16083  we have studied the effects of curare on respo...

Total passages after cleaning: (27975, 1)


In [4]:
import re
import pandas as pd

# Load original passages
passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

print("Before Preprocessing Sample:\n", passages.head(3))

# 1. Remove duplicates
passages = passages.drop_duplicates(subset=['passage'])

# 2. Clean text
def clean_passage(text):
    text = text.strip().lower()  # Lowercase
    text = re.sub(r"[^a-z0-9.,;:()\-/% ]", " ", text)  # Keep important punctuation
    text = re.sub(r"\s+", " ", text)  # Remove multiple spaces
    return text

passages['passage'] = passages['passage'].astype(str).apply(clean_passage)

# 3. Drop empty passages
passages = passages[passages['passage'].str.strip() != ""]

# 4. Show final result
print("\nAfter Preprocessing Sample:\n", passages.head(3))
print("\nTotal passages after cleaning:", passages.shape)

# 5. Save the cleaned passages
passages.to_csv("processed_passages.csv", index=True)
print("\nSaved cleaned passages as 'processed_passages.csv'")


Before Preprocessing Sample:
                                                  passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...

After Preprocessing Sample:
                                                  passage
id                                                      
9797   new data on viruses isolated from patients wit...
11906  we describe an improved method for detecting d...
16083  we have studied the effects of curare on respo...

Total passages after cleaning: (27975, 1)

Saved cleaned passages as 'processed_passages.csv'


In [5]:
!pip install transformers datasets evaluate bert-score rouge-score -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.4 MB/s eta 0:00:00


# Model Setup

In [6]:
# Load Flan-T5 (a strong open-source instruction-tuned model)
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a pipeline for QA
chatbot = pipeline("text2text-generation", model=model, tokenizer=tokenizer)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


In [7]:
# --- Install dependencies ---
!pip install transformers datasets evaluate bert-score rouge-score -q

# --- Imports ---
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import re

# --- Load the model ---
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
chatbot = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# --- Load processed passages ---
passages = pd.read_csv("processed_passages.csv", index_col=0)

# --- Few-Shot Prompt Builder ---
def build_prompt(question, context):
    few_shot_examples = """
Q: Is the protein Papilin secreted?
A: Yes, Papilin is a secreted protein.

Q: Are long non coding RNAs spliced?
A: Long non-coding RNAs appear to be spliced through similar mechanisms as mRNA.

"""
    return f"{few_shot_examples}Context: {context}\nQ: {question}\nA:"

# --- Chatbot Response Function ---
def chatbot_response(question):
    # Simple keyword search to get context
    keyword = question.split()[0].lower()
    matches = passages[passages['passage'].str.contains(keyword, na=False)]

    if matches.empty:
        return "Sorry, I don’t have the answer to that."

    context = " ".join(matches['passage'].head(3))
    prompt = build_prompt(question, context)
    response = chatbot(prompt, max_length=256, do_sample=False)
    return response[0]['generated_text']

# --- Test the chatbot ---
question = "Is Hirschsprung disease a mendelian or a multifactorial disorder?"
print("Q:", question)
print("A:", chatbot_response(question))


Device set to use cuda:0
Token indices sequence length is longer than the specified maximum sequence length for this model (665 > 512). Running this sequence through the model will result in indexing errors


Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Hirschsprung disease is a multifactorial disorder.


In [8]:
pip install faiss-cpu transformers sentence-transformers bert-score rouge-score lime


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 48.2 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=dc748a2b6a6c0d76fb8890f84915b324ef68a74f0b59f06e9cf227eab5f67f05
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [9]:
!pip install transformers datasets evaluate bert_score rouge_score --quiet

from transformers import pipeline

# Load pretrained QA model
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cuda:0


# Loading Test data

In [10]:
import pandas as pd

# Load QA data (adjust the path if necessary)
qa_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

print("QA Data loaded:", qa_df.shape)
print(qa_df.columns)
print(qa_df.head(3))


QA Data loaded: (4719, 3)
Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  are: epidermal growt...   
2                 Yes,  papilin is a secreted protein   

                                 relevant_passage_ids  
id                                                     
0   [20598273, 6650562, 15829955, 15617541, 230011...  
1   [23821377, 24323361, 23382875, 22247333, 23787...  
2   [21784067, 19297413, 15094122, 7515725, 332004...  


In [11]:
def join_passages(passage_ids_str):
    try:
        passage_ids = eval(passage_ids_str)
    except:
        passage_ids = []
    texts = []
    for pid in passage_ids:
        if pid in passages_df.index:
            p = passages_df.loc[pid, 'passage']
            # Convert to string if not None/NaN
            if pd.notna(p):
                texts.append(str(p))
    return " ".join(texts)


In [13]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate


In [14]:
model_checkpoint = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)


In [15]:
!pip install transformers datasets evaluate bert_score rouge_score --quiet


In [16]:
!rm -r ~/.cache/huggingface/datasets


rm: cannot remove '/root/.cache/huggingface/datasets': No such file or directory


In [17]:
!pip install --upgrade pip
!pip install datasets transformers evaluate bert_score rouge_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [23]:
import fsspec

fs = fsspec.filesystem("hf")
files = fs.ls("datasets/rag-datasets/rag-mini-bioasq/data/")
print(files)


[{'name': 'datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet', 'size': 0, 'type': 'directory', 'tree_id': '67386e3f0c71b5818b6d10bfb67b1cdb8dc558a8', 'last_commit': LastCommitInfo(oid='16861628138912b7571fb262d25a94bbef8e4daa', title='add dataset', date=datetime.datetime(2023, 10, 28, 14, 56, 50, tzinfo=datetime.timezone.utc))}, {'name': 'datasets/rag-datasets/rag-mini-bioasq/data/test.parquet', 'size': 0, 'type': 'directory', 'tree_id': '38c0a3060cd490c117a67cfd77da7908ffce8d19', 'last_commit': LastCommitInfo(oid='463a10473ab7e68eeb4a150b4825601f8496a81f', title='before creating dataset', date=datetime.datetime(2023, 10, 28, 9, 15, 27, tzinfo=datetime.timezone.utc))}]


In [26]:
import pandas as pd

# Load the final preprocessed data (replace the path with your actual file path)
final_df = pd.read_csv('/content/processed_passages.csv')

# Check the loaded data
print(final_df.shape)
print(final_df.head())


(27975, 2)
      id                                            passage
0   9797  new data on viruses isolated from patients wit...
1  11906  we describe an improved method for detecting d...
2  16083  we have studied the effects of curare on respo...
3  23188  kinetic and electrophoretic properties of 230-...
4  23469  male wistar specific-pathogen-free rats aged 2...


In [34]:
!pip install --upgrade datasets fsspec


  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)


# Loading Test questions for preview

In [4]:
import pandas as pd

# Load test questions from HuggingFace Hub
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Preview available columns
print(test_df.columns)
print(test_df.head())


Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   
3                   Are long non coding RNAs spliced?   
4                   Is RANKL secreted from the cells?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  are: epidermal growt...   
2                 Yes,  papilin is a secreted protein   
3   Long non coding RNAs appear to be spliced thro...   
4   Receptor activator of nuclear factor κB ligand...   

                                 relevant_passage_ids  
id                                                     
0   [20598273, 665

In [5]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
import pandas as pd

# Load test data
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Use a biomedical QA model from Hugging Face
model_name = "ktrapeznikov/biobert_v1.1_pubmed_squad_v2"
qa_pipeline = pipeline("question-answering", model=model_name, tokenizer=model_name)

# Hardcoded example context (use real passages later if doing RAG)
dummy_context = "Papilin is a protein involved in ECM remodeling and is secreted."

# Example usage
sample = test_df.iloc[2]  # "Is the protein Papilin secreted?"
result = qa_pipeline(question=sample["question"], context=dummy_context)

print("Question:", sample["question"])
print("Model Answer:", result["answer"])
print("Gold Answer:", sample["answer"])


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at ktrapeznikov/biobert_v1.1_pubmed_squad_v2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Question: Is the protein Papilin secreted?
Model Answer: secreted
Gold Answer: Yes,  papilin is a secreted protein


In [8]:
from transformers import BertTokenizer, BertForQuestionAnswering
import torch

# Load model and tokenizer
model_name = "ktrapeznikov/biobert_v1.1_pubmed_squad_v2"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForQuestionAnswering.from_pretrained(model_name)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


Some weights of the model checkpoint at ktrapeznikov/biobert_v1.1_pubmed_squad_v2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

In [9]:
def answer_question(question, model, tokenizer, device):
    inputs = tokenizer(question, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits
    start_index = torch.argmax(start_logits)
    end_index = torch.argmax(end_logits)
    answer_ids = inputs["input_ids"][0][start_index : end_index + 1]
    return tokenizer.decode(answer_ids, skip_special_tokens=True)


In [10]:
df["predicted_answer"] = df["question"].apply(lambda q: answer_question(q, model, tokenizer, device))


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [12]:
question = "Is the protein Papilin secreted?"


In [13]:
inputs = tokenizer(
    question,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    padding="max_length"
)


In [14]:
pip install transformers datasets tqdm

# Load the pretrained model and run the batch inference on all questions

In [15]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForQuestionAnswering
from tqdm import tqdm

# Load the test dataset directly from HuggingFace Hub
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Load BioBERT QA model and tokenizer
model_name = "ktrapeznikov/biobert_v1.1_pubmed_squad_v2"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForQuestionAnswering.from_pretrained(model_name)

# Setup device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def batch_answer_questions(questions, model, tokenizer, device, batch_size=16):
    all_answers = []
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_questions = questions[i : i + batch_size]
        inputs = tokenizer(
            batch_questions,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
        for j in range(len(batch_questions)):
            start_idx = torch.argmax(start_logits[j])
            end_idx = torch.argmax(end_logits[j]) + 1
            answer_ids = inputs['input_ids'][j][start_idx:end_idx]
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
            all_answers.append(answer.strip())
    return all_answers

# Run batch inference on all questions
questions = test_df['question'].tolist()
test_df['predicted_answer'] = batch_answer_questions(questions, model, tokenizer, device, batch_size=16)

# Show sample predictions
for idx in range(5):
    print(f"Q: {test_df.loc[idx, 'question']}")
    print(f"Predicted: {test_df.loc[idx, 'predicted_answer']}")
    print(f"Gold answer: {test_df.loc[idx, 'answer']}")
    print("-" * 40)


Some weights of the model checkpoint at ktrapeznikov/biobert_v1.1_pubmed_squad_v2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 295/295 [00:18<00:00, 16.03it/s]

Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Predicted: multifactorial
Gold answer: Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.
----------------------------------------
Q: List signaling molecules (ligands) that interact with the receptor EGFR?
Predicted: 
Gold answer: The 7 known EGFR ligands  are: epidermal growth factor (EGF), betacellulin (BTC), epiregulin (EPR), heparin-binding EGF (HB-EGF), transforming growth factor-α [TGF-α], amphiregulin (AREG) and epigen (EPG).
----------------------------------------
Q: Is the protein Papilin secreted?
Predicted: 
Gold answer: Yes,  papilin is a secreted protei

In [16]:
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")


In [18]:
print(passages_df.columns)
print(passages_df.head())


Index(['passage'], dtype='object')
                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
23188  Kinetic and electrophoretic properties of 230-...
23469  Male Wistar specific-pathogen-free rats aged 2...


In [19]:
passage_dict = passages_df['passage'].to_dict()  # keys are the index 'id', values are the text


In [20]:
def get_context_text(passage_ids):
    context = ""
    for pid in passage_ids[:3]:
        context += passage_dict.get(pid, "") + " "
    return context.strip()


In [21]:
questions = test_df['question'].tolist()
contexts = test_df['relevant_passage_ids'].apply(get_context_text).tolist()


In [22]:
def batch_answer_question_context(questions, contexts, model, tokenizer, device, batch_size=8):
    all_answers = []
    model.eval()
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_q = questions[i : i + batch_size]
        batch_c = contexts[i : i + batch_size]

        inputs = tokenizer(
            batch_q,
            batch_c,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        for j in range(len(batch_q)):
            start_idx = torch.argmax(start_logits[j])
            end_idx = torch.argmax(end_logits[j]) + 1
            answer_ids = inputs['input_ids'][j][start_idx:end_idx]
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
            all_answers.append(answer.strip())

    return all_answers


In [23]:
predicted_answers = batch_answer_question_context(questions, contexts, model, tokenizer, device, batch_size=8)
test_df['predicted_answer'] = predicted_answers


100%|██████████| 590/590 [00:18<00:00, 31.93it/s]


In [24]:
for idx in range(5):
    print(f"Q: {test_df.loc[idx, 'question']}")
    print(f"Predicted: {test_df.loc[idx, 'predicted_answer']}")
    print(f"Gold answer: {test_df.loc[idx, 'answer']}")
    print("-" * 40)


Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Predicted: multifactorial
Gold answer: Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.
----------------------------------------
Q: List signaling molecules (ligands) that interact with the receptor EGFR?
Predicted: 
Gold answer: The 7 known EGFR ligands  are: epidermal growth factor (EGF), betacellulin (BTC), epiregulin (EPR), heparin-binding EGF (HB-EGF), transforming growth factor-α [TGF-α], amphiregulin (AREG) and epigen (EPG).
----------------------------------------
Q: Is the protein Papilin secreted?
Predicted: 
Gold answer: Yes,  papilin is a secreted protei

In [25]:
def get_context_text(passage_ids):
    context = ""
    for pid in passage_ids[:1]:  # use only top 1 passage
        context += passage_dict.get(pid, "") + " "
    return context.strip()


In [32]:
def get_context_text(passage_ids):
    context = ""
    for pid in passage_ids[:1]:  # Use only top 1 passage
        context += passage_dict.get(pid, "") + " "
    return context.strip()

def batch_answer_question_context(questions, contexts, model, tokenizer, device, batch_size=8):
    all_answers = []
    model.eval()
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_q = questions[i : i + batch_size]
        batch_c = contexts[i : i + batch_size]

        inputs = tokenizer(
            batch_q,
            batch_c,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        for j in range(len(batch_q)):
            start_idx = torch.argmax(start_logits[j])
            end_idx = torch.argmax(end_logits[j]) + 1
            answer_ids = inputs['input_ids'][j][start_idx:end_idx]
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
            if not answer:
                answer = "I don't know"
            all_answers.append(answer)

    return all_answers


In [34]:
few_shot_examples = [
    ("Is Hirschsprung disease a mendelian or a multifactorial disorder?",
     "Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."),
    ("Is the protein Papilin secreted?",
     "Yes, papilin is a secreted protein."),
    ("Are long non coding RNAs spliced?",
     "Long non coding RNAs appear to be spliced through the same pathway as the mRNAs."),
]


In [35]:
passage_dict = passages_df['passage'].to_dict()


In [36]:
def get_context_text(passage_ids, max_passages=3):
    context = ""
    for pid in passage_ids[:max_passages]:
        context += passage_dict.get(pid, "") + " "
    return context.strip()


In [37]:
def build_prompt(question, context, few_shot_examples):
    prompt = ""
    for q, a in few_shot_examples:
        prompt += f"Q: {q}\nA: {a}\n\n"
    prompt += f"Context: {context}\n\nQ: {question}\nA:"
    return prompt


In [38]:
def generate_answer(prompt, model, tokenizer, device, max_length=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=max_length)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.strip()


In [40]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


# prompt tuning

In [41]:
from tqdm import tqdm

predicted_answers = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    passage_ids = row['relevant_passage_ids']
    context = get_context_text(passage_ids)
    prompt = build_prompt(question, context, few_shot_examples)
    answer = generate_answer(prompt, model, tokenizer, device)
    predicted_answers.append(answer)

test_df['predicted_answer'] = predicted_answers


100%|██████████| 4719/4719 [1:08:51<00:00,  1.14it/s]


In [42]:
for idx in range(5):
    print(f"Q: {test_df.loc[idx, 'question']}")
    print(f"Predicted: {test_df.loc[idx, 'predicted_answer']}")
    print(f"Gold answer: {test_df.loc[idx, 'answer']}")
    print("-" * 40)


Q: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Predicted: Hirschsprung disease is a multifactorial disorder.
Gold answer: Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.
----------------------------------------
Q: List signaling molecules (ligands) that interact with the receptor EGFR?
Predicted: EGFR is a protein that is encoded by the EGFR gene.
Gold answer: The 7 known EGFR ligands  are: epidermal growth factor (EGF), betacellulin (BTC), epiregulin (EPR), heparin-binding EGF (HB-EGF), transforming growth factor-α [TGF-α], amphiregulin (AREG) and epigen (EPG).
----------------------------------------
Q: Is th

In [44]:
!pip install evaluate


# Evaluated metrics for ROUGE-L and BERTScore (F1)

In [46]:
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Compute ROUGE-L
rouge_result = rouge.compute(
    predictions=test_df["predicted_answer"].tolist(),
    references=test_df["answer"].tolist(),
    use_stemmer=True
)

# Compute BERTScore (F1)
bertscore_result = bertscore.compute(
    predictions=test_df["predicted_answer"].tolist(),
    references=test_df["answer"].tolist(),
    lang="en"
)

# Display results
print(f"ROUGE-L Score: {rouge_result['rougeL']:.4f}")
print(f"BERTScore F1: {sum(bertscore_result['f1']) / len(bertscore_result['f1']):.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE-L Score: 0.2106
BERTScore F1: 0.8509


In [49]:
!pip install --upgrade transformers


In [6]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
model.save_pretrained("/content/my_model_checkpoint")
tokenizer.save_pretrained("/content/my_tokenizer_checkpoint")


('/content/my_tokenizer_checkpoint/tokenizer_config.json',
 '/content/my_tokenizer_checkpoint/special_tokens_map.json',
 '/content/my_tokenizer_checkpoint/vocab.txt',
 '/content/my_tokenizer_checkpoint/added_tokens.json',
 '/content/my_tokenizer_checkpoint/tokenizer.json')

In [52]:
!pip uninstall -y transformers
!pip install transformers --upgrade


Found existing installation: transformers 4.53.3
Uninstalling transformers-4.53.3:
  Successfully uninstalled transformers-4.53.3
  Using cached transformers-4.53.3-py3-none-any.whl.metadata (40 kB)
Using cached transformers-4.53.3-py3-none-any.whl (10.8 MB)


In [8]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

model_path = "/content/my_model_checkpoint"
tokenizer_path = "/content/my_tokenizer_checkpoint"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForQuestionAnswering.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)


In [10]:
from transformers import BertTokenizer, BertForQuestionAnswering

model = BertForQuestionAnswering.from_pretrained("/content/my_model_checkpoint")
tokenizer = BertTokenizer.from_pretrained("/content/my_tokenizer_checkpoint")
